### Zero-shot klasifikacija studenata

In [ ]:
import pandas as pd
import joblib

df = pd.read_csv("../data/processed/final.csv")
model = joblib.load("../models/final_model.pkl")
threshold = joblib.load("../models/threshold.pkl")

In [2]:
df["target"] = df["final_result"].isin(["Fail", "Withdrawn"]).astype(int)
df["year"] = df["code_presentation"].astype(str).str[:4]

test_df = df[df["year"] == "2014"].copy()
X_test = test_df.drop(columns=["final_result", "target", "id_student", "year"],errors="ignore")
y_test = test_df["target"]

In [3]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= threshold).astype(int)

test_df["risk_probability"] = y_proba
test_df["ml_prediction"] = y_pred
test_df[["id_student", "code_module","code_presentation","final_result","risk_probability", "ml_prediction"]].head()

,id_student,code_module,code_presentation,final_result,risk_probability,ml_prediction
358,6516,AAA,2014J,Pass,0.265252,0
359,24734,AAA,2014J,Pass,0.831003,1
360,26192,AAA,2014J,Distinction,0.050245,0
361,28061,AAA,2014J,Pass,0.182801,0
362,31600,AAA,2014J,Pass,0.327259,0


In [4]:
student = test_df[test_df["ml_prediction"] == 1].iloc[0]
student

code_module                                    AAA
code_presentation                            2014J
id_student                                   24734
gender                                           F
region                                South Region
highest_education               Lower Than A Level
imd_band                                    60-70%
age_band                                      0-35
num_of_prev_attempts                             0
studied_credits                                 60
disability                                       N
final_result                                  Pass
avg_days_before_deadline_CMA                   0.0
avg_days_before_deadline_TMA                  -7.0
avg_score_CMA                                  0.0
avg_score_TMA                                 39.5
count_submitted_CMA                            0.0
count_submitted_TMA                            2.0
std_score_CMA                                  0.0
std_score_TMA                  

In [5]:
student_opis = f"""
Student pohađa kurs {student["code_module"]} u prezentaciji/semestru {student["code_presentation"]}.
Prosečna TMA ocena studenta je {student["avg_score_TMA"]}.
Student je predao {student["count_submitted_TMA"]} od ukupno {student["total_assessments_TMA"]} TMA zadataka.
Stopa predaje TMA zadataka je {student["submission_rate_TMA"]}.
Prosečan broj dana pre roka za TMA zadatke je {student["avg_days_before_deadline_TMA"]}.

Prosečna CMA ocena studenta je {student["avg_score_CMA"]}.
Student je predao {student["count_submitted_CMA"]} od ukupno {student["total_assessments_CMA"]} CMA zadataka.
Stopa predaje CMA zadataka je {student["submission_rate_CMA"]}.
Prosečan broj dana pre roka za CMA zadatke je {student["avg_days_before_deadline_CMA"]}.

Ukupan broj klikova na VLE platformi je {student["total_clicks"]}.
Poslednji dan aktivnosti studenta je {student["last_active_day"]}.
Broj različitih aktivnosti na platformi je {student["unique_activities"]}.
"""
print(student_opis)


Student pohađa kurs AAA u prezentaciji/semestru 2014J.
Prosečna TMA ocena studenta je 39.5.
Student je predao 2.0 od ukupno 2.0 TMA zadataka.
Stopa predaje TMA zadataka je 1.0.
Prosečan broj dana pre roka za TMA zadatke je -7.0.

Prosečna CMA ocena studenta je 0.0.
Student je predao 0.0 od ukupno 0.0 CMA zadataka.
Stopa predaje CMA zadataka je 0.0.
Prosečan broj dana pre roka za CMA zadatke je 0.0.

Ukupan broj klikova na VLE platformi je 189.0.
Poslednji dan aktivnosti studenta je 86.0.
Broj različitih aktivnosti na platformi je 6.0.



In [6]:
from transformers import pipeline

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
candidate_labels = ["rizičan student","uspešan student"]
hf_rez = zero_shot_classifier(
    student_opis,
    candidate_labels=candidate_labels
)
hf_rez

{'sequence': '\nStudent pohađa kurs AAA u prezentaciji/semestru 2014J.\nProsečna TMA ocena studenta je 39.5.\nStudent je predao 2.0 od ukupno 2.0 TMA zadataka.\nStopa predaje TMA zadataka je 1.0.\nProsečan broj dana pre roka za TMA zadatke je -7.0.\n\nProsečna CMA ocena studenta je 0.0.\nStudent je predao 0.0 od ukupno 0.0 CMA zadataka.\nStopa predaje CMA zadataka je 0.0.\nProsečan broj dana pre roka za CMA zadatke je 0.0.\n\nUkupan broj klikova na VLE platformi je 189.0.\nPoslednji dan aktivnosti studenta je 86.0.\nBroj različitih aktivnosti na platformi je 6.0.\n',
 'labels': ['uspešan student', 'rizičan student'],
 'scores': [0.5532909631729126, 0.4467090666294098]}

In [8]:
hf_rezultat = pd.DataFrame({
    "klasa": hf_rez["labels"],
    "score": hf_rez["scores"]
})

hf_rezultat

,klasa,score
0,uspešan student,0.553291
1,rizičan student,0.446709


In [9]:
ml_klasa = "rizičan student" if student["ml_prediction"] == 1 else "uspešan student"
hf_klasa = hf_rez["labels"][0]

print("Klasifikacija ML modela:", ml_klasa)
print("Verovatnoća rizika ML modela:", round(student["risk_probability"], 4))
print("Klasifikacija Hugging Face modela:", hf_klasa)
print("Score Hugging Face modela:", round(hf_rez["scores"][0], 4))

Klasifikacija ML modela: rizičan student
Verovatnoća rizika ML modela: 0.831
Klasifikacija Hugging Face modela: uspešan student
Score Hugging Face modela: 0.5533
